In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
import joblib
import os

In [9]:
# 1. Cargar el dataset limpio
csv_path = '../data/clean/players_clean.csv'
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"No se encontró el archivo en {csv_path}")

df = pd.read_csv(csv_path)

print("⚙️ Procesando características para el modelo...")

⚙️ Procesando características para el modelo...


In [11]:
# Convertir fecha de nacimiento a Edad (Año actual de referencia: 2026)
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce')
df['age'] = 2026 - df['date_of_birth'].dt.year
df['age'] = df['age'].fillna(df['age'].median())

In [13]:
# Rellenar alturas faltantes con la mediana
df['height_in_cm'] = df['height_in_cm'].fillna(df['height_in_cm'].median())

In [14]:
# Mapear las posiciones categóricas (texto) a números discretos
position_map = {'Attack': 1, 'Midfield': 2, 'Defender': 3, 'Goalkeeper': 4, 'Missing': 0}
df['position_encoded'] = df['position'].map(position_map).fillna(0)

In [15]:
# Asegurar que highest_market_value tenga valores válidos
df['highest_market_value_in_eur'] = df['highest_market_value_in_eur'].fillna(0)

In [16]:
# 2. Selección de Variables (Features y Target)
features = ['age', 'height_in_cm', 'highest_market_value_in_eur', 'position_encoded']
X = df[features]
y = df['market_value_in_eur'].fillna(0) # Lo que queremos predecir

In [17]:
# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
# Crear directorio de modelos si no existe
os.makedirs('models', exist_ok=True)

In [19]:
# 3. Entrenar y guardar los 3 Modelos de Ensemble
modelos = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
}

print("\n🚀 Iniciando entrenamiento de los modelos de Ensemble...")
for nombre, modelo in modelos.items():
    print(f"⏱️ Entrenando {nombre}...")
    modelo.fit(X_train, y_train)
    
    # Evaluar la precisión con R² Score
    score_train = modelo.score(X_train, y_train)
    score_test = modelo.score(X_test, y_test)
    print(f"   📊 {nombre} -> Train R²: {score_train:.4f} | Test R²: {score_test:.4f}")
    
    # Guardar en archivo físico
    filename = f"models/{nombre.lower().replace(' ', '_')}_model.pkl"
    joblib.dump(modelo, filename)
    print(f"   💾 Guardado como '{filename}'\n")

print("¡Todos los modelos han sido entrenados y exportados con éxito!")


🚀 Iniciando entrenamiento de los modelos de Ensemble...
⏱️ Entrenando Random Forest...
   📊 Random Forest -> Train R²: 0.9749 | Test R²: 0.8466
   💾 Guardado como 'models/random_forest_model.pkl'

⏱️ Entrenando Gradient Boosting...
   📊 Gradient Boosting -> Train R²: 0.9651 | Test R²: 0.8345
   💾 Guardado como 'models/gradient_boosting_model.pkl'

⏱️ Entrenando XGBoost...
   📊 XGBoost -> Train R²: 0.9574 | Test R²: 0.8455
   💾 Guardado como 'models/xgboost_model.pkl'

¡Todos los modelos han sido entrenados y exportados con éxito!
